In [1]:
# Paramètres modifiables
# Excentricité
e = 0.8
# Nombre de corps sur l'orbite
Nc = 2
# Taille du secteur
Ns = 100
# Vitesse
speed = 5
#-----------------------------------------------------------------------
# Bibliothèques utilisées
import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as ani
import matplotlib.patches as ptc
from scipy.integrate import odeint
import itertools as itt
#-----------------------------------------------------------------------
# Quelques constantes
au = 149597870700.0  # m (pas km!) distance moyenne terre soleil
G = 6.6742e-11  # SI
MS = 1.989e30  # kg, Soleil

def Foo(e):
    return (1+e)/(1-e**2)**0.5 #corrige vitesse initiale ne fonction de e 

# Conditions initiales corrigées
C0 = [(1+e)*au, 0, 0, 0, 6.283*au/(365.25*86400)/Foo(e), 0] #part du periphelie x = 1 + e et avec vy corrigée et le reste nul

# Calcul des dérivées
def Der(Y, t): 

    '''Entrée : position et vitesse du corps à un instant donné

Sortie : les dérivées (vitesses et accélérations)'''
    x, y, z, vx, vy, vz = Y
    r3 = (x**2 + y**2 + z**2)**1.5
    ax = -G * MS * x / r3
    ay = -G * MS * y / r3
    az = -G * MS * z / r3
    return [vx, vy, vz, ax, ay, az] 

# Simulation
T = np.linspace(0.0, 86400*365*2, 1260*2) # ici T vaut 2 ans 
res = odeint(Der,    C0, T) #resultat integration a tout temps 

#-----------------------------------------------------------------------
# Configuration de matplotlib
matplotlib.use('TkAgg')  # Backend compatible avec l'animation

# Création de la figure
fig, ax = plt.subplots(figsize=(10, 10))

# Animation des résultats
ax.plot([0], [0], "yo", markersize=20, label="Soleil") #soleil au cnetr e
ax.plot(res[:1260, 0], res[:1260, 1], 'k--', linewidth=0.5, label="Orbite")

def GetCoords(i, mask=False):
    idx = i % 1260
    end_idx = min(idx + Ns, 1260)
    T = np.array([[0] + list(res[idx:end_idx, 0]), 
                  [0] + list(res[idx:end_idx, 1])]).T #recupere coordonnée pour tracer aire 
    if mask:
        T *= 0
    return T

# Construit la liste des polygones et corps célestes
polys = []
colors = 'ygbrmc'
for i in range(Nc):
    offset = i * 1260 // Nc #pour ne pas avoir les corps au même endroit   au depart
    color = colors[i % 6]
    poly = ptc.Polygon(GetCoords(0, True), closed=True, color=color, alpha=0.3)
    corps, = ax.plot([0], [0], color + 'o', markersize=8)
    polys.append((offset, poly, corps))
    ax.add_patch(poly)

ax.axis("equal")
ax.set_title("Deuxième loi de Kepler : vitesse aréolaire constante", fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Variables pour la détection de changement de taille
size_cache = [None, None]

def SizeChanged(ax): #pour redimensionner image 
    current = [ax.bbox.width, ax.bbox.height]
    if size_cache != current:
        size_cache[:] = current
        return True
    return False

def Update(i):
    artists = []
    for k, poly, corps in polys:
        crds = GetCoords(k + i)
        poly.set_xy(crds)
        corps.set_data([crds[-1, 0]], [crds[-1, 1]])
        artists.extend([poly, corps])
    
    if SizeChanged(ax):
        fig.canvas.draw_idle()
    return artists

def Init():
    artists = []
    for k, poly, corps in polys:
        crds = GetCoords(k, True)
        poly.set_xy(crds)
        corps.set_data(np.ma.array([crds[-1, 0]], mask=True),
                       np.ma.array([crds[-1, 1]], mask=True))
        artists.extend([poly, corps])
    return artists

anim = ani.FuncAnimation(fig, Update, frames=range(0, 1260, int(speed)),
                         interval=50, blit=True, init_func=Init, repeat=True, cache_frame_data=False)

# Affichage
plt.tight_layout()
plt.show()